# Track A — Whole-line design optimisation results

Loads `results/track_a/outer_summary.json` produced by `run_optimize.py` and visualises:

1. Outer objective curve $\mathcal{L}^\star(N^B)$ and $\mathcal{L}^\star(N^B) + \lambda_N N^B$
2. Optimal $\boldsymbol{\theta}_c^\star$ components vs $N^B$
3. At global $N^{B\star}$: slack heatmap $S_{i,k}$ and per-segment marginals
4. Sanity checks

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yaml
from pathlib import Path

from ksb.simulation.ksb_simulation import KSBSimulation
from ksb.analysis.cost import compute_S_bb, compute_Phi_bb

RESULTS_DIR = Path('..') / 'results' / 'track_a'
CONFIG_DIR  = Path('..') / 'configs'

with open(RESULTS_DIR / 'outer_summary.json') as f:
    summary = json.load(f)

with open(CONFIG_DIR / f"{summary['config']}.yaml") as f:
    base_cfg = yaml.safe_load(f)

lambda_N = summary['lambda_N']
lambda_U = summary['lambda_U']
lambda_L = summary['lambda_L']
lambda_T = summary['lambda_T']
nb_star  = summary['nb_star']

nb_vals    = sorted(int(k) for k in summary['inner_results'])
L_star     = np.array([summary['inner_results'][str(nb)]['L_star']    for nb in nb_vals])
outer_vals = np.array([summary['outer_scores'][str(nb)]               for nb in nb_vals])

print(f"Config      : {summary['config']}")
print(f"λ_U={lambda_U}  λ_L={lambda_L}  λ_T={lambda_T}  λ_N={lambda_N}")
print(f"N^B range   : {nb_vals[0]} .. {nb_vals[-1]}")
print(f"N^B★        : {nb_star}")
print(f"Wall time   : {summary['total_wall_time_s']:.1f} s")

## 1. Outer objective curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
fig.suptitle(r'Outer objective vs $N^B$', fontsize=13)

ax.plot(nb_vals, L_star,     color='C0', linewidth=1.8, marker='o', ms=5,
        label=r'$\mathcal{L}^\star(N^B)$')
ax.plot(nb_vals, outer_vals, color='C1', linewidth=1.8, marker='s', ms=5, linestyle='--',
        label=r'$\mathcal{L}^\star + \lambda_N N^B$')

# mark global optimum
nb_idx  = nb_vals.index(nb_star)
ax.axvline(nb_star, color='C3', linestyle=':', linewidth=1.5)
ax.scatter([nb_star], [outer_vals[nb_idx]], color='C3', s=120, zorder=5,
           label=f'$N^{{B\\star}} = {nb_star}$')

ax.set_xlabel(r'$N^B$ (number of buffer segments)', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_xticks(nb_vals)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.show()

## 2. Optimal $\boldsymbol{\theta}_c^\star$ components vs $N^B$

In [ ]:
theta_keys = [
    ('L_buffer',   r'$L^B$ (m)',        'C0'),
    ('beta',       r'$\beta$ (skew)',    'C1'),
    ('gamma',      r'$\gamma$ (curv.)',  'C2'),
    ('v_buff_out', r'$v^{BR}$ (m/s)',    'C3'),
    ('a_u_max',    r'$a_u^\max$ (m/s²)', 'C4'),
    ('v_u_max',    r'$v_u^\max$ (m/s)', 'C5'),
    ('eta_s',      r'$\eta_s$',          'C6'),
    ('eta_r',      r'$\eta_r$',          'C7'),
]

fig, axes = plt.subplots(2, 4, figsize=(14, 6), constrained_layout=True)
fig.suptitle(r'Optimal $\boldsymbol{\theta}_c^\star$ vs $N^B$', fontsize=13)

vd = float(base_cfg['slot_rate_ppm']) / 60.0 * float(base_cfg['slot_length'])

for ax, (key, label, color) in zip(axes.ravel(), theta_keys):
    vals = [summary['inner_results'][str(nb)]['theta_star'][key] for nb in nb_vals]
    ax.plot(nb_vals, vals, color=color, linewidth=1.6, marker='o', ms=4)
    ax.axvline(nb_star, color='C3', linestyle=':', linewidth=1.2, alpha=0.7)

    # reference lines
    if key == 'v_buff_out':
        ax.axhline(vd, color='gray', linestyle='--', linewidth=0.9,
                   label=f'$v_d$ = {vd:.2f}')
        ax.legend(fontsize=8)
    if key in ('beta', 'gamma'):
        ax.axhline(0.0, color='gray', linestyle=':', linewidth=0.8)

    ax.set_xlabel(r'$N^B$', fontsize=10)
    ax.set_ylabel(label, fontsize=10)
    ax.set_xticks(nb_vals[::2])
    ax.grid(axis='y', alpha=0.25)

plt.show()

# Print starred values
ts = summary['inner_results'][str(nb_star)]['theta_star']
print(f"\nAt N^B★ = {nb_star}:")
for k, lbl, _ in theta_keys:
    print(f"  {k:12s} = {ts[k]:.4f}")

## 3. Slack heatmap and per-segment marginals at $N^{B\star}$

In [ ]:
# Reconstruct optimal cfg
opt_result = summary['inner_results'][str(nb_star)]
ts = opt_result['theta_star']

opt_cfg = dict(base_cfg)
opt_cfg['n_buffer_seg']   = nb_star
opt_cfg['L_buffer']       = ts['L_buffer']
opt_cfg['beta']           = ts['beta']
opt_cfg['gamma']          = ts['gamma']
opt_cfg['v_buff_out']     = ts['v_buff_out']
opt_cfg['a_u_max']        = ts['a_u_max']
opt_cfg['v_u_max']        = ts['v_u_max']
opt_cfg['slot_length']    = ts['slot_length']
opt_cfg['slot_rate_ppm']  = ts['slot_rate_ppm']
opt_cfg['j_u_max']        = float(base_cfg.get('jmax', 100.0))

j_max = float(opt_cfg['jmax'])

SEEDS = [0, 1, 2, 3, 42]
S_list = []

for seed in SEEDS:
    result = KSBSimulation(cfg=opt_cfg).run(seed=seed)
    if result.segment_events is not None:
        S = compute_S_bb(result.segment_events, j_max)
        S_list.append(S)

# Stack and take mean over seeds for the heatmap (representative picture)
# Show the first seed's heatmap and marginals across all seeds
S_first = S_list[0]     # (b-1, N_B)
S_all   = np.vstack(S_list)  # (n_seeds*(b-1), N_B)

print(f"Slack matrix shape (first seed): {S_first.shape}")
print(f"Negative cells (first seed): {np.sum(S_first < 0)} / {S_first.size}")
print(f"Negative cells (all seeds):  {np.sum(S_all < 0)} / {S_all.size}")

In [ ]:
fig = plt.figure(figsize=(14, 5), constrained_layout=True)
gs  = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[2, 1.2])
fig.suptitle(
    fr'Slack $S_{{i,k}}$ at $N^{{B\star}}={nb_star}$  '
    fr'(seed={SEEDS[0]},  batch={opt_cfg["batch"]})',
    fontsize=13,
)

# ── Heatmap ─────────────────────────────────────────────────────────────────
ax_heat = fig.add_subplot(gs[0])
vmax = np.percentile(np.abs(S_first), 95) if S_first.size > 0 else 1.0
vmax = max(vmax, 0.01)
im = ax_heat.imshow(
    S_first, aspect='auto', origin='lower',
    cmap='RdYlGn', vmin=-vmax, vmax=vmax,
    interpolation='nearest',
)
plt.colorbar(im, ax=ax_heat, label='Slack $S_{i,k}$ (s)')
ax_heat.set_xlabel('Segment $k$', fontsize=11)
ax_heat.set_ylabel('Pair $i$', fontsize=11)
ax_heat.set_xticks(np.arange(nb_star))
ax_heat.set_xticklabels([str(k + 1) for k in range(nb_star)])
ax_heat.set_title('Red = infeasible ($S < 0$),  Green = slack', fontsize=9, color='gray')

# ── Per-segment marginals ────────────────────────────────────────────────────
ax_box = fig.add_subplot(gs[1])
data_by_seg = [S_all[:, k] for k in range(nb_star)]
bp = ax_box.boxplot(
    data_by_seg,
    vert=True, patch_artist=True,
    medianprops=dict(color='black', linewidth=1.5),
)
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, nb_star))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax_box.axhline(0.0, color='gray', linestyle='--', linewidth=1.0)
ax_box.set_xlabel('Segment $k$', fontsize=11)
ax_box.set_ylabel('Slack $S_{i,k}$ (s)', fontsize=11)
ax_box.set_xticks(np.arange(1, nb_star + 1))
ax_box.set_xticklabels([str(k + 1) for k in range(nb_star)])
ax_box.set_title(f'All seeds ({len(S_list)} × {opt_cfg["batch"]-1} pairs)', fontsize=9, color='gray')

plt.show()

## 4. Sanity checks

In [ ]:
vd_check = float(opt_cfg['slot_rate_ppm']) / 60.0 * float(opt_cfg['slot_length'])
v_br     = ts['v_buff_out']
sentinel  = opt_result['sentinel']

n_neg   = np.sum(S_all < 0)
n_total = S_all.size

print("=== Sanity checks ===")
print(f"  v^BR* = {v_br:.3f}  ≥  v_d = {vd_check:.3f} ?  {v_br >= vd_check}")
print(f"  sentinel=False ?  {not sentinel}")
print(f"  Negative slack cells (all seeds): {n_neg} / {n_total}  "
      f"({100.*n_neg/n_total:.1f}%)")
print()
print("=== Optimal parameters ===")
for k, lbl, _ in theta_keys:
    print(f"  {k:12s} = {ts[k]:.4f}")
print(f"  {'gamma sign':12s} : {'< 0 (middle-heavy ✓)' if ts['gamma'] < 0 else '≥ 0 (edge-heavy)'}")
print(f"  {'eta_r':12s} : {'< 2.0 (not at upper bound ✓)' if ts['eta_r'] < 1.98 else '= 2.0 (upper bound — λ_T too small?)'}") 

if n_neg == 0:
    print("\n  All slack cells ≥ 0 — design is feasible under bang-bang primitive.")
else:
    print(f"\n  {n_neg} infeasible cells — design has residual violations; consider larger λ_U or wider bounds.")